**DATA CLEANING NOTES PER DATASET**

1. Gemini Dataset
    - A bit easy to clean if we were to base the cleaning on the

2. Claude Dataset
    - Needs a bit of cleaning since it has a lot of curly braces
    - It has equations and code 
    - to clean: remaining greek operators, fractions, and code

3. MGTBench Dataset (ChatGPT)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

4. MGTBench Dataset (Human-written)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

5. BAWE Corpus Dataset
    - Needs the most cleaning since it an HTML file 
    - Need to convert it into an HTML file so we can clean it properly

In [1]:
# Import the necessary libraries for cleaning the data
import os
import re
import pandas as pd
import numpy as numpy
from pathlib import Path
from tqdm import tqdm
import ftfy
import ast
from langdetect import detect, detect_langs, DetectorFactory, LangDetectException

pd.set_option('display.max_colwidth', 150)
tqdm.pandas(desc="Cleaning Text")

print("Libraries has been imported!")

Libraries has been imported!


In [2]:
# Establish the directories where the data will be read and stored after processing
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
DATA_DIR = PROJECT_ROOT / "data"

# Directories of the Raw Datasets
RAW_AI_DIR = DATA_DIR / "raw" / "ai"
RAW_HUMAN_DIR = DATA_DIR / "raw" / "human"

# Directories of the Processed Datasets where it will be stored after cleaning the data
PROCESSED_AI_DIR = DATA_DIR / "processed" / "ai"
PROCESSED_HUMAN_DIR = DATA_DIR / "processed" / "human"

# To ensure that the out processed directories exist
PROCESSED_AI_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_HUMAN_DIR.mkdir(parents=True, exist_ok=True)

print("Data Cleaning Paths Ready:")
print(f"  Reading AI Raw Data:        {RAW_AI_DIR.resolve()}")
print(f"  Reading Human Raw Data:     {RAW_HUMAN_DIR.resolve()}")
print(f"  Saving AI Processed Data:   {PROCESSED_AI_DIR.resolve()}")
print(f"  Saving Human Processed Data:{PROCESSED_HUMAN_DIR.resolve()}")


Data Cleaning Paths Ready:
  Reading AI Raw Data:        C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\ai
  Reading Human Raw Data:     C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\human
  Saving AI Processed Data:   C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai
  Saving Human Processed Data:C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\human


**METHODS FOR CLEANING DATA**

- Cleans math texts
- Cleans texts that has citations
- Cleans texts that has code in it
- Strips reference lists
- Cleans texts that has any numberings in it
- Checks whether a text is creative works and it will remove that since creatives is not considered as an academic text

In [3]:
from pandas._libs import interval

def clean_math_texts(text):
    """
    Cleans math equations, LaTeX expressions, and Unicode mathematical symbols from text
    by replacing them with [[EQUATION]] placeholders.
    """
    if not isinstance(text, str):
        return text

    # Normalize literal escaped newlines and citation markers
    text = text.replace('\\n', ' ')
    text = re.sub(r'\[\d+\]', '', text)

    # LaTeX environments
    text = re.sub(r'\\begin\{[a-zA-Z0-9\*]+\}.*?\\end\{[a-zA-Z0-9\*]+\}', ' [[EQUATION]] ', text, flags=re.DOTALL)

    # LaTeX block and display math
    text = re.sub(r'\$\$.*?\$\$', ' [[EQUATION]] ', text, flags=re.DOTALL)
    text = re.sub(r'\\\[.*?\\\]', ' [[EQUATION]] ', text, flags=re.DOTALL)

    # LaTeX inline math
    text = re.sub(r'\$([^\$\n]+)\$', ' [[EQUATION]] ', text)
    text = re.sub(r'\\\((.*?)\\\)', ' [[EQUATION]] ', text)

    # Common LaTeX commands
    text = re.sub(r'\\[a-zA-Z]+(\{.*?\})*', ' [[EQUATION]] ', text)

    # Integrals with bounds and differentials
    text = re.sub(r'[∮∫][₀-₉⁰-⁹\^]*[^\.\n]*?d[A-Za-z]+', ' [[EQUATION]] ', text)

    # Algebraic equations containing '=' or comparison operators
    text = re.sub(r'\b[a-zA-Z0-9\(\)\{\}\[\]\+\-\*/\^]+\s*(?:<=|>=|!=|==|=|<|>|≠|≤|≥|≈)\s*[a-zA-Z0-9\(\)\{\}\[\]\+\-\*/\^\s\._]+', ' [[EQUATION]] ', text)

    # Exponents and derivatives
    text = re.sub(r'\b[a-zA-Z0-9\(\)]+\^[a-zA-Z0-9\(\)\+\-]+\b', ' [[EQUATION]] ', text)
    text = re.sub(r'\bd[A-Za-z]/d[A-Za-z]\b', ' [[EQUATION]] ', text)

    # "n choose k" style combinatorics notation
    text = re.sub(r'\([a-zA-Z0-9\s\+\-]+choose[a-zA-Z0-9\s\+\-]+\)', '[[EQUATION]]', text)

    # Catches sentences that survived token-level cleaning but are still mostly fragments/placeholders
    cleaned_sentences = []
    sentences = re.split(r'(?<=[.!?])\s+', text)
    prev_was_equation = False

    for sent in sentences:
        placeholder_count = sent.count('[[EQUATION]]')
        non_placeholder_text = re.sub(r'\[\[EQUATION\]\]', '', sent)
        word_count = len(re.findall(r'[a-zA-Z]{3,}', non_placeholder_text))

        # If a sentence has 2+ placeholders and very few real words around them,
        # it's fragment soup, so we need to collapse to a single [[EQUATION]]
        is_fragment_heavy = placeholder_count >= 2 and word_count < 6

        if is_fragment_heavy or (placeholder_count >= 1 and word_count == 0):
            if not prev_was_equation:
                cleaned_sentences.append("[[EQUATION]]")
            prev_was_equation = True
        else:
            cleaned_sentences.append(sent)
            prev_was_equation = False

    text = ' '.join(cleaned_sentences)
    text = re.sub(r'(\[\[EQUATION\]\]\s*){2,}', '[[EQUATION]]', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def clean_residual_math_noise(text):
    """
        Final mop-up pass for math-heavy content that survives the main
        clean_math_texts regex pipeline: standalone Unicode math symbols,
        Pascal's-triangle-style bare numeric sequences, ASCII lattice/diagram
        art, and orphaned brackets left over from earlier substitutions.
        Run this AFTER clean_math_texts in the pipeline.
    """
    if not isinstance(text, str):
        return text

    GREEK= 'Σ∂ΦθΘεδ∇ΔαβγλμπΩω'

    # For catching functions like f(x), g(x), etc
    for _ in range(2):
        text = re.sub(
            r'\b[a-zA-Z]{1,2}\d{0,2}\([a-zA-Z0-9,\s\+\-\*/\^\.]*\)',
            ' [[EQUATION]] ', text
        )

    # Units with exponents 
    text = re.sub(r'\b[a-zA-Z]+(?:\*[a-zA-Z]+)?/[a-zA-Z]+[\^²³]*\d*\b', ' [[EQUATION]] ', text)

    # For absolute values
    text = re.sub(r'\|[a-zA-Z0-9\+\-\*/\^\.,\s]{1,40}\|', ' [[EQUATION]] ', text)

    # For replacing square roots
    text = re.sub(r'√\s*\(?[a-zA-Z0-9\+\-\*/\^\.,\s]*\)?', ' [[EQUATION]] ', text)

    # For the Unicode of fraction and superscript characters attached to a term
    text = re.sub(r'[a-zA-Z0-9\)]*[½⅓¼¾⅔⅕⅖⅗][a-zA-Z0-9\(]*', ' [[EQUATION]] ', text)
    text = re.sub(r'\b[a-zA-Z0-9\)]+[²³¹⁰⁴⁵⁶⁷⁸⁹]+', ' [[EQUATION]] ', text)

    # Plus-minus: ±b, ± 1
    text = re.sub(r'±\s*[a-zA-Z0-9\.\(\)]+', ' [[EQUATION]] ', text)

    # Greek-letter variables and comparison chains
    text = re.sub(
        rf'(?<![a-zA-Z0-9])[{GREEK}][a-zA-Z0-9]*(?:\s*(?:[+\-*/^=<>]|<=|>=)\s*(?:(?<![a-zA-Z0-9])[{GREEK}]|[a-zA-Z0-9])[a-zA-Z0-9\.]*)*',
        ' [[EQUATION]] ', text
    )

    # Partial-derivative fraction pattern: [[EQUATION]]f1/[[EQUATION]]x 
    text = re.sub(r'\[\[EQUATION\]\]\s*[a-zA-Z0-9]*\s*/\s*\[\[EQUATION\]\]\s*[a-zA-Z0-9]*', ' [[EQUATION]] ', text)

    # Stepped-subscript variables: xn+1, yn+1, zn+1, x0, y0
    # Restricted to x/y/z and the "n+1"/"n-1" pattern specifically, to
    # avoid false-positive matches on ordinary words like "in", "an", "on".
    text = re.sub(r'\b(?:x|y|z)(?:n\+1|n-1|0)\b', ' [[EQUATION]] ', text)
    text = re.sub(r'\b[a-zA-Z]n\+1\b|\b[a-zA-Z]n-1\b', ' [[EQUATION]] ', text)

    # Bracket-wrapped variable/component
    # Negative lookaround guards prevent matching our own [[TAG]] placeholders
    text = re.sub(
        rf'(?<!\[)\[(?:[{GREEK}][a-zA-Z0-9]*|[a-zA-Z]\d)(?:[\s,][{GREEK}a-zA-Z0-9]*)*\](?!\])',
        ' [[EQUATION]] ', text
    )

    # Bare numeric sequences 
    text = re.sub(r'(?:\b\d+\b[\s,]+){3,}\b\d+\b', ' [[EQUATION]] ', text)

    # Orphaned brackets/parenthesis left behind. Runs before the function-name
    # merge below, since nested calls like g(f(x)) need their parenthesis
    # absorbed first before "g" can be merged into the resulting tag
    for _ in range(3):
        text = re.sub(r'\([^()]*\[\[EQUATION\]\][^()]*\)', ' [[EQUATION]] ', text)
        text = re.sub(r'(?<!\[)\[[^\[\]]*\[\[EQUATION\]\][^\[\]]*\](?!\])', ' [[EQUATION]] ', text)
    
    # Merge a leftover math-function-name letter sitting right before a placeholder
    text = re.sub(r'\b[fghFGH]\d{0,2}\s*\[\[EQUATION\]\]', ' [[EQUATION]] ', text)

    # Stray single bracket immediately touching our own [[ / ]] delimiters
    text = re.sub(r'\[\s*(?=\[\[EQUATION\]\])', '', text)
    text = re.sub(r'(?<=\[\[EQUATION\]\])\s*\]', '', text)

    # Consolidate and clean whitespace
    text = re.sub(r'(\[\[EQUATION\]\]\s*){2,}', '[[EQUATION]] ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def strip_reference_list(text):
    """Cuts off everything from the first 'Sources:' / 'References' """

    match = re.search(r'\n?(Sources|References|Bibliography):', text, flags=re.IGNORECASE)

    if match:
        return text[:match.start()].strip()

    return text


def clean_code_texts(text):
    """
        Cleans code blocks and inline code snippets from text by replacing them 
        with [[CODE]] placeholders.
    """

    if not isinstance(text, str):
        return text

    # Replace markdown fenced code blocks
    text = re.sub(r'```[a-zA-Z0-9_\+\#-]*\n?[\s\S]*?```', ' <CODE> ', text)

    # Replace inline code snippets
    text = re.sub(r'`[^`\n]+`', ' [[CODE]] ', text)

    # Consolidate consecutive [[CODE]] tags and normalize whitespace
    text = re.sub(r'(\[\[CODE\]\]\s*){2,}', '[[CODE]] ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


def clean_citations(text):
    """
        Replaces any citations whether intext to [[CITATION]]
    """
    if not isinstance(text, str):
        return text

    # Bracketed numeric citations: [1], [12], [1,2]
    text = re.sub(r'\[\d+(,\s*\d+)*\]', ' [[CITATION]] ', text)

    # Parenthetical citations: (Smith, 2020), (Smith et al., 2020), (Smith & Jones, 2020),
    # (Smith, 2020; Lee, 2019), (Smith 2020) — comma optional, semicolon-joined multiples
    text = re.sub(
        r'\([A-Z][a-zA-Z\.\s,&]*?(?:et al\.)?\s*,?\s*\d{4}[a-z]?(?:\s*;\s*[A-Z][a-zA-Z\.\s,&]*?\d{4}[a-z]?)*\)',
        ' [[CITATION]] ', text
    )

    # Narrative citations: "Smith (2020)", "Smith et al. (2020)"
    text = re.sub(r'\b[A-Z][a-zA-Z]+(?:\set al\.)?\s\(\d{4}[a-z]?\)', ' [[CITATION]] ', text)

    text = re.sub(r'(\[\[CITATION\]\]\s*){2,}', '[[CITATION]] ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def clean_complexity_notation(text):
    """
        Cleans the rows that has any Big-O notation in the dataset
    """

    if not isinstance(text, str):
        return text

    # Big-O, Big-Theta, Big-Omega notation: O(n), O(log n), O(n^2), Θ(n), Ω(n log n)
    # Requires the letter to be standalone (not preceded by another letter) to avoid
    # false matches like "info(x)" or "to(n)" — also drops lowercase 'o' since it's
    # too ambiguous with common English words even with the lookbehind guard.
    text = re.sub(r'(?<![a-zA-Z])[OΘΩ]\(\s*[a-zA-Z0-9\s\^\+\-\*/,]*\)', ' [[COMPLEXITY]] ', text)

    text = re.sub(r'(\[\[COMPLEXITY\]\]\s*){2,}', '[[COMPLEXITY]] ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def clean_list_numbering(text):
    """
        Cleans texts that has any numbering in it
        e.g. 1. 2. ...
    """

    if not isinstance(text, str):
        return text

    # Numbered list markers: "1.", "2.", "\n3."
    text = re.sub(r'(?:^|(?<=[\s:;\.]))\d{1,2}\.\s+(?=[A-Za-z])', ' ', text)

    # Letter list markers
    text = re.sub(r'(?:^|(?<=[\s:;\.]))[a-zA-Z][\.\)]\s+(?=[A-Za-z0-9])', ' ', text)

    # Roman Numerals list markers
    text = re.sub(r'(?:^|(?<=[\s:;\.]))(?:i{1,3}|iv|v|vi{0,3}|ix|x)[\.\)]\s+', ' ', text, flags=re.IGNORECASE)

    # Dash/bullet markers: "- Puts more money...", "\n- Allows workers..."
    text = re.sub(r'(?:^|\n)\s*-\s+', ' ', text)

    # Cleans up any whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def clean_url(text):
    """
        Replaces URLs (even with or without a preceding hyperlink label) with
        [[URL]] placeholders.
    """
    if not isinstance(text, str):
        return text

    # Standard http(s):// URLs
    text = re.sub(r'https?://\S+', ' [[URL]] ', text)

    # www.-prefixed URLs without a scheme
    text = re.sub(r'\bwww\.\S+', ' [[URL]] ', text)

    text = re.sub(r'(\[\[URL\]\]\s*){2,}', '[[URL]] ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def is_academic_content(prompt="", text=""):
    """
        This function returns False if either the prompt of the text contains a creative,
        fictional, commercial script, or stage-direction markers that fall outside the
        academic scope
    """

    combined = (str(prompt) + " " + str(text)).lower()

    creative_keywords = [
        'script', 'commercial', 'advertisement', 'ad script', 'screenplay', 
        'short story', 'story', 'novel', 'fiction', 'fairy tale', 'fairytale', 
        'poem', 'poetry', 'lyrics', 'song', 'monologue', 'dialogue between', 
        'playwright', 'haiku', 'sonnet', 'screenwriter', 'broadway', 'fanfiction',
        'imagine you are', 'shapeshift', 'shapeshifting', 'roleplay', 'role-play',
        'pretend you are', 'you wake up as', 'from the perspective of a',
        'paint the vivid picture', 'paint a vivid picture', 'vivid picture of',
        'epic battle', 'describe in vivid detail',
        "captivating the reader's imagination", 'captivate the reader',
        'lore behind', 'board game', 'video game'
    ]

    if any(kw in combined for kw in creative_keywords):
        return False

    # Stage Direction & Script Structural Markers in response texts
    script_patterns = [
        r'\[visual:', r'\[audio:', r'\[music', r'\[sfx:', r'\[scene', 
        r'\[camera', r'\[upbeat', r'\[fade', r'\bnarrator:', r'\bint\.\s', r'\bext\.\s'
    ]

    # This catches the entertainment/hobbyist framing specifically
    non_academic_framing = [
        "captivating the reader's imagination", 'captivate the reader',
        'lore behind', 'immersive world of', 'storied island', 'embark on an'
    ]

    if any(re.search(pattern, combined) for pattern in script_patterns):
        return False

    return True

foreign_skip_counter = {"too_short": 0}

def clean_foreign_script(text):
    """
        Replaces runs of non-Latin script characters (e.g. Chinese, Japanese,
        Korean, Arabic, Cyrillic) with [[FOREIGN]] placeholders, since spaCy's
        English pipeline and ELECTRA's English tokenizer can't meaningfully
        process non-English script.
    """

    if not isinstance(text, str):
        return text

    # Unicode ranges of the languages Chinese, Japanese, Korean, Arabic, and Cyrillic
    text = re.sub(
        r'[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af\u0600-\u06ff\u0400-\u04ff]+',
        ' [[FOREIGN]] ', text
    )

    # Since French/Spanish detection are sentence-level, the use latin script and
    # and these can't be matched by Unicode ranges like how different languages can
    sentences = re.split(r'(?<=[.!?])\s+', text)
    cleaned_sentences = []

    for sent in sentences:
        # Only run the langauge detection on sentencs with enough content
        # and langdetect is unreliable on very short strings
        if len(sent.strip()) > 15:
            try:
                lang = detect(sent)
                if lang in ('fr', 'es'):
                    cleaned_sentences.append("[[FOREIGN]]")
                    continue
            except LangDetectException:
                foreign_skip_counter["too_short"] += 1
        else:
            foreign_skip_counter["too_short"] += 1
            
        cleaned_sentences.append(sent)
    
    text = ' '.join(cleaned_sentences)
    text = re.sub(r'(\[\[FOREIGN\]\]\s*){2,}', '[[FOREIGN]] ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

Since most datasets have a lot of placeholders which can render the dataset unsable/useful for both spaCy and ELECTRA. 

In [4]:
def placeholder_density(text):
    """
        This method is meant to remove the rows that have too many
        tokens on the cleaned data
    """

    if not isinstance(text, str):
        return 0

    placeholder_count = len(re.findall(r'\[\[EQUATION\]\]|\[\[CODE\]\]|\[\[CITATION\]\]|\[\[COMPLEXITY\]\]|\[\[URL\]\]|\[\[FOREIGN\]\]', text))
    word_count = len(re.findall(r'\b[a-zA-Z]{2,}\b', text))
    total = placeholder_count + word_count

    return placeholder_count / total if total > 0 else 0


def placeholder_density_windowed(text, window_chars=150, threshold=0.5):
    """
        Catches locally dense clusters of placeholders even inside one long
        run-on sentence (where the whole-document placeholder_density() can
        stay low because of lots of surrounding prose elsewhere). Slides a
        fixed-size character window across the text and flags True if ANY
        window exceeds the local density threshold — meant to be used
        alongside (not instead of) placeholder_density() when deciding
        whether to drop a row.
    """
    if not isinstance(text, str) or len(text) < window_chars:
        return False

    tag_pattern = re.compile(r'\[\[[A-Z]+\]\]')

    for start in range(0, len(text) - window_chars, window_chars // 2):
        window = text[start:start + window_chars]
        tag_char_len = sum(len(m.group()) for m in tag_pattern.finditer(window))
        if tag_char_len / len(window) >= threshold:
            return True

    return False


def clean_pipeline(text):
    """
        Combined cleaning pipeline for all of the datasets
    """

    text = clean_code_texts(text)
    text = clean_url(text)
    text = clean_foreign_script(text)
    text = clean_complexity_notation(text)
    text = strip_reference_list(text)
    text = clean_citations(text)
    text = clean_list_numbering(text)
    text = clean_math_texts(text)
    text = clean_residual_math_noise(text)

    return text

In [5]:
import IPython.display as ipd

def extract_claude_prompt_and_response(text):
    """
        Extracts human prompt and claude/gpt response specifically from 
        Claude dataset's dictionary-style conversation strings.
        Handles plain text datasets by returning an empty prompt and raw text.
    """
    if not isinstance(text, str):
        return "", str(text)

    if text.strip().startswith('[') and "'from'" in text and "'value'" in text:
        try:
            data = ast.literal_eval(text)
            prompt = ""
            raw_response = ""

            for turn in data:
                if isinstance(turn, dict):
                    role = turn.get('from')
                    val = turn.get('value', '')

                    if role == 'human' and not prompt:
                        prompt = val
                    elif role in ('gpt', 'assistant'):
                        raw_response += val + " "

            return prompt.strip(), raw_response.strip()

        # Fallback regex incase it encountered a problem in parsing
        except Exception:
            prompt_match = re.search(r"\{'from':\s*'human',\s*'value':\s*\"\"?(.*?)\"\"?\}", text, flags=re.DOTALL)
            resp_match = re.search(r"\{'from':\s*'(?:gpt|assistant)',\s*'value':\s*\"\"?(.*?)\"\"?\}", text, flags=re.DOTALL)
            prompt = prompt_match.group(1) if prompt_match else ""
            raw_response = resp_match.group(1) if resp_match else text

            return prompt.strip(), raw_response.strip()

    return "", text.strip()


def clean_claude_dataset(claude_csv_path, sample_size=200, density_threshold=0.4):
    """
        Cleans the Claude AI dataset, filters out non-academic creative prompts,
        extracts prompt and cleaned response, filters out placeholder-dense rows
        (both whole-document density and locally dense run-on clusters),
        counts tag insertions, displays summary & sample tables, and saves
        output to PROCESSED_AI_DIR.
    """
    if not claude_csv_path.exists():
        print(f"File not found at: {claude_csv_path}")
        return None

    # Reset the foreign-skip counter at the start of each run so counts
    # don't accumulate across repeated calls in the same session
    foreign_skip_counter["too_short"] = 0

    print(f"Loading {'first ' + str(sample_size) if sample_size else 'all'} rows from {claude_csv_path.name}...")
    df_raw = pd.read_csv(claude_csv_path, nrows=sample_size)
    prompts = []
    cleaned_responses = []
    dropped_creative = 0
    dropped_density = 0
    dropped_locally_dense = 0

    print("Cleaning & filtering Claude dataset...")
    for raw_text in tqdm(df_raw['conversations'], desc="Processing Rows"):
        prompt, raw_response = extract_claude_prompt_and_response(str(raw_text))

        # Filter out non-academic creative writing prompts (stories, scripts, poems)
        if not is_academic_content(prompt, raw_response):
            dropped_creative += 1
            continue

        cleaned_resp = clean_pipeline(raw_response)

        # Filter out rows that are mostly placeholder tokens after cleaning
        # (too notation-dense to yield meaningful stylometric features)
        density = placeholder_density(cleaned_resp)
        if density >= density_threshold:
            dropped_density += 1
            continue

        # Filter out rows with a locally dense cluster of placeholders even
        # if the whole-document density looks fine (e.g. a flattened matrix
        # description sitting inside one long run-on sentence, surrounded
        # by otherwise-normal prose elsewhere in the same response)
        if placeholder_density_windowed(cleaned_resp):
            dropped_locally_dense += 1
            continue

        prompts.append(prompt)
        cleaned_responses.append(cleaned_resp)

    # Build clean output DataFrame with prompt and cleaned_text columns
    df_processed = pd.DataFrame({
        'prompt': prompts,
        'cleaned_text': cleaned_responses
    })

    summary_data = {
        "Metric": [
            "Total Rows Loaded",
            "Dropped (Non-Academic/Creative)",
            "Dropped (Too Placeholder-Dense)",
            "Dropped (Locally Dense Cluster)",
            "Total Academic Rows Kept",
            "[[EQUATION]] Tags Inserted",
            "[[CODE]] Tags Inserted",
            "[[CITATION]] Tags Inserted",
            "[[COMPLEXITY]] Tags Inserted",
            "[[URL]] Tags Inserted",
            "[[FOREIGN]] Tags Inserted",
            "Sentences Skipped (Too Short to Detect Language)"
        ],
        "Count": [
            len(df_raw),
            dropped_creative,
            dropped_density,
            dropped_locally_dense,
            len(df_processed),
            df_processed['cleaned_text'].str.count(r'\[\[EQUATION\]\]').sum(),
            df_processed['cleaned_text'].str.count(r'\[\[CODE\]\]').sum(),
            df_processed['cleaned_text'].str.count(r'\[\[CITATION\]\]').sum(),
            df_processed['cleaned_text'].str.count(r'\[\[COMPLEXITY\]\]').sum(),
            df_processed['cleaned_text'].str.count(r'\[\[URL\]\]').sum(),
            df_processed['cleaned_text'].str.count(r'\[\[FOREIGN\]\]').sum(),
            foreign_skip_counter["too_short"]
        ]
    }
    df_summary = pd.DataFrame(summary_data)

    print("\n--- CLEANING SUMMARY ---")
    ipd.display(df_summary)

    # Save processed data to PROCESSED_AI_DIR
    filename = f"claude_dataset_cleaned_{sample_size}.csv" if sample_size else "claude_dataset_cleaned.csv"
    output_path = PROCESSED_AI_DIR / filename
    df_processed.to_csv(output_path, index=False)
    print(f"\nSuccessfully saved cleaned dataset ({len(df_processed)} rows) to:\n  {output_path.resolve()}")

    # Display first 20 rows table (prompt & cleaned_text)
    print("\n--- SAMPLE CLEANED DATA (FIRST 20 ROWS) ---")
    ipd.display(df_processed.head(20))

    return df_processed


claude_dataset = RAW_AI_DIR / 'claude_dataset.csv'
df_cleaned_claude = clean_claude_dataset(claude_dataset, sample_size=200)

Loading first 200 rows from claude_dataset.csv...
Cleaning & filtering Claude dataset...


Processing Rows: 100%|██████████| 200/200 [00:06<00:00, 32.70it/s]


--- CLEANING SUMMARY ---


,Metric,Count
0,Total Rows Loaded,200
1,Dropped (Non-Academic/Creative),127
2,Dropped (Too Placeholder-Dense),2
3,Dropped (Locally Dense Cluster),8
4,Total Academic Rows Kept,63
5,[[EQUATION]] Tags Inserted,215
6,[[CODE]] Tags Inserted,67
7,[[CITATION]] Tags Inserted,35
8,[[COMPLEXITY]] Tags Inserted,4
9,[[URL]] Tags Inserted,2



Successfully saved cleaned dataset (63 rows) to:
  C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\claude_dataset_cleaned_200.csv

--- SAMPLE CLEANED DATA (FIRST 20 ROWS) ---


,prompt,cleaned_text
0,Create a function in Python code that accepts a string input and outputs the number of vowels in the string. The function signature and a sample c...,"Here's a Python function that counts the number of vowels in a given string: < [[EQUATION]] : We define a set of vowels, including both lowercase ..."
1,"Write a well-articulated series of paragraphs explaining how to effectively structure one's writing through the use of topic sentences, transition...","To effectively structure one's writing, it's important to employ clear topic sentences, smooth transitions, a consistent theme, and proper punctua..."
2,Derive a proof for the Pythagorean Theorem relating the sides of a right triangle and use it to help students understand and remember the key math...,Certainly! I'll provide a proof for the Pythagorean Theorem and then discuss how to help students understand and remember this important mathemati...
3,•Faraday's experiments on induction which established the basic principles. Discuss how Faraday's observations lead to the concept of magnetic fl...,"Faraday's Experiments and Magnetic Flux: In the 1830s, Michael Faraday conducted a series of experiments that established the fundamental principl..."
4,Imagine that you have two claims that appear to conflict with one another. Analyze the reasons and evidence provided for both claims. Look for way...,Thank you for this interesting thought exercise. I'll do my best to analyze and reconcile two hypothetical conflicting claims in a reasoned way. S...
5,Summarize the U.S court case Miranda v. Arizona (1966) in which the Supreme Court established that the police must inform individuals of their Mir...,Miranda [[CITATION]] was a landmark U. Supreme Court case that established important constitutional protections for criminal suspects during polic...
6,"Write a short Python program that generates 10 random integer numbers between 1 and 100, calculates the sum and average of those numbers, and pri...",Here's a short Python program that does what you've described: < [[EQUATION]] : It imports the [[CODE]] module to generate random numbers. It uses...
7,"Explain in detail how interest rates set by central banks impact economic growth and stability. Consider factors like investment, consumption, emp...","Interest rates set by central banks play a crucial role in shaping economic growth and stability. These rates, often referred to as the ""policy ra..."
8,Explain in simple terms how a country's interest rate impacts investment and economic growth. Focus on providing concrete examples and avoid jargon.,"A country's interest rate, set by its central bank, has a significant impact on investment and economic growth. When interest rates are low, it be..."
9,"Write a step-by-step procedure that thoroughly explains how you would debug an error in a Python program, covering multiple possible causes of and...","Here's a comprehensive step-by-step procedure for debugging errors in a Python program, covering multiple error types, causes, solutions, and debu..."


In [6]:
def clean_claude_dataset(dataset):
    
    pass

In [7]:
def clean_mgtbench_human_dataset(dataset):

    pass

In [15]:
import re
import pandas as pd
import IPython.display as ipd


# Placeholder types used by the cleaning pipeline
PLACEHOLDER_NAMES = (
    "EQUATION",
    "CODE",
    "CITATION",
    "COMPLEXITY",
    "URL",
    "FOREIGN"
)

PLACEHOLDER_PATTERN = (
    r"\[\[(?:EQUATION|CODE|CITATION|COMPLEXITY|URL|FOREIGN)\]\]"
)


# Replace code blocks and inline code with a placeholder
def clean_code_texts(text):
    if not isinstance(text, str):
        return text
    text = re.sub(
        r"```[\s\S]*?```",
        " [[CODE]] ",
        text)

    text = re.sub(
        r"`[^`\n]+`",
        " [[CODE]] ",
        text)
    return text


# Replace URLs with a placeholder
def clean_url(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " [[URL]] ",
        text,
        flags=re.IGNORECASE)
    return text


# Replace foreign-language scripts with a placeholder
def clean_foreign_script(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"[\u0400-\u04FF]+",
        " [[FOREIGN]] ",
        text)
    text = re.sub(
        r"[\u0600-\u06FF]+",
        " [[FOREIGN]] ",
        text
    )
    text = re.sub(
        r"[\u4E00-\u9FFF]+",
        " [[FOREIGN]] ",
        text
    )
    text = re.sub(
        r"[\u3040-\u30FF]+",
        " [[FOREIGN]] ",
        text
    )
    text = re.sub(
        r"[\uAC00-\uD7AF]+",
        " [[FOREIGN]] ",
        text
    )
    return text


# Replace common algorithm complexity notation
def clean_complexity_notation(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"\bO\s*\([^)]{1,100}\)",
        " [[COMPLEXITY]] ",
        text
    )
    text = re.sub(
        r"\bΩ\s*\([^)]{1,100}\)",
        " [[COMPLEXITY]] ",
        text
    )
    text = re.sub(
        r"\bΘ\s*\([^)]{1,100}\)",
        " [[COMPLEXITY]] ",
        text
    )

    return text


# Replace common academic citation formats
def clean_citations(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"\[\s*\d+(?:\s*[-,]\s*\d+)*\s*\]",
        " [[CITATION]] ",
        text
    )

    text = re.sub(
        r"\\(?:cite|citep|citet|citealp|citeauthor|citeyear)"
        r"(?:\[[^\]]*\])?"
        r"\{[^}]*\}",
        " [[CITATION]] ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\([A-Z][A-Za-z.\-]+"
        r"(?:\s+et al\.)?"
        r",?\s*\d{4}[a-z]?\)",
        " [[CITATION]] ",
        text
    )

    text = re.sub(
        r"\b[A-Z][A-Za-z.\-]+"
        r"(?:\s+et al\.)?"
        r"\s*\(\d{4}[a-z]?\)",
        " [[CITATION]] ",
        text
    )

    return text


# Detect large LaTeX equations and replace them with one placeholder
def collapse_complete_equations(text):
    if not isinstance(text, str):
        return text

    equation_pattern = re.compile(
        r"(?<!\w)"
        r"[\(\[]?\s*"
        r"(?:\\{1,2})"
        r"(?:"
        r"frac|"
        r"dfrac|"
        r"tfrac|"
        r"sqrt|"
        r"sum|"
        r"prod|"
        r"int|"
        r"oint|"
        r"lim|"
        r"mathcal|"
        r"mathrm|"
        r"mathbf|"
        r"mathit|"
        r"operatorname|"
        r"partial|"
        r"nabla"
        r")"
        r"[^.!?\n]{0,2000}",
        flags=re.IGNORECASE
    )

    def replace_equation(match):
        equation = match.group(0)

        has_equals = "=" in equation

        has_latex = bool(
            re.search(
                r"\\{1,2}[A-Za-z]+",
                equation
            )
        )

        has_math_structure = bool(
            re.search(
                r"[\^_{}]",
                equation
            )
        )

        if has_latex and (
            has_equals or has_math_structure
        ):
            return " [[EQUATION]] "

        return equation

    text = equation_pattern.sub(
        replace_equation,
        text
    )

    return text


# Clean LaTeX, math notation, and equation symbols
def clean_math_texts(text):
    if not isinstance(text, str):
        return text

    text = text.replace("\\n", " ")
    text = text.replace("\\t", " ")
    text = text.replace("\r", " ")
    text = text.replace("\n", " ")

    text = collapse_complete_equations(text)

    text = re.sub(
        r"\\begin\{[^}]+\}[\s\S]*?\\end\{[^}]+\}",
        " [[EQUATION]] ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\$\$[\s\S]*?\$\$",
        " [[EQUATION]] ",
        text
    )

    text = re.sub(
        r"\\\[[\s\S]*?\\\]",
        " [[EQUATION]] ",
        text
    )

    text = re.sub(
        r"\\\([\s\S]*?\\\)",
        " [[EQUATION]] ",
        text
    )

    text = re.sub(
        r"\$(?!\$)[^\$\n]+?\$(?!\$)",
        " [[EQUATION]] ",
        text
    )

    text = re.sub(
        r"\\(?:"
        r"ref|eqref|pageref|label|"
        r"section|subsection|subsubsection|paragraph|"
        r"textbf|textit|texttt|text|"
        r"mathrm|mathbf|mathit|mathsf|mathtt|"
        r"operatorname|"
        r"frac|dfrac|tfrac|sqrt|"
        r"sum|prod|int|oint|lim|"
        r"partial|nabla"
        r")"
        r"\*?"
        r"(?:\s*\{[^{}]*\})+",
        " [[EQUATION]] ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\\[A-Za-z]+\*?(?:\{[^{}]*\})?",
        " [[EQUATION]] ",
        text
    )

    text = re.sub(
        r"\\[A-Za-z]+",
        " [[EQUATION]] ",
        text
    )

    math_symbols = (
        "∑∏∫∮√∞≈≠≤≥±×÷"
        "∂∇∆∈∉⊂⊃⊆⊇"
        "∀∃∄"
        "→←↔⇒⇐⇔"
    )

    text = re.sub(
        "[" + re.escape(math_symbols) + "]",
        " [[EQUATION]] ",
        text
    )

    return text


# Remove everything after a References/Bibliography section
def strip_reference_list(text):
    if not isinstance(text, str):
        return text

    match = re.search(
        r"(?:^|\n)\s*"
        r"(?:References|Bibliography|Sources)"
        r"\s*:?\s*",
        text,
        flags=re.IGNORECASE
    )

    if match:
        text = text[:match.start()]

    return text


# Remove list numbering while keeping the actual text
def clean_list_numbering(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"(?<!\w)\d{1,2}[.)]\s+(?=[A-Za-z])",
        " ",
        text
    )

    text = re.sub(
        r"(?<!\w)[A-Za-z][.)]\s+(?=[A-Za-z])",
        " ",
        text
    )

    text = re.sub(
        r"(?<!\w)"
        r"(?:i{1,3}|iv|v|vi{0,3}|ix|x)"
        r"[.)]\s+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"[•●▪◦]\s*",
        " ",
        text
    )

    return text


# Remove leftover LaTeX characters and mathematical noise
def clean_residual_math_noise(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"\\[A-Za-z]+",
        " ",
        text
    )

    text = re.sub(
        r"[{}]",
        " ",
        text
    )

    text = re.sub(
        r"\\+",
        " ",
        text
    )

    text = text.replace(
        "~",
        " "
    )

    text = re.sub(
        r"(?<!\w)[_^]+(?!\w)",
        " ",
        text
    )

    text = re.sub(
        r"([.,;:!?])\1+",
        r"\1",
        text
    )

    return text


# Remove parentheses surrounding placeholders
def clean_placeholder_parentheses(text):
    if not isinstance(text, str):
        return text

    previous = None

    while previous != text:
        previous = text

        text = re.sub(
            rf"\(\s*({PLACEHOLDER_PATTERN})\s*\)",
            r"\1",
            text
        )

    return text


# Make all placeholder formats consistent
def normalize_placeholders(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"<EQUATION>",
        "[[EQUATION]]",
        text
    )

    text = re.sub(
        r"<CODE>",
        "[[CODE]]",
        text
    )

    text = re.sub(
        r"<CITATION>",
        "[[CITATION]]",
        text
    )

    text = re.sub(
        r"<COMPLEXITY>",
        "[[COMPLEXITY]]",
        text
    )

    text = re.sub(
        r"<URL>",
        "[[URL]]",
        text
    )

    text = re.sub(
        r"<FOREIGN>",
        "[[FOREIGN]]",
        text
    )

    text = re.sub(
        r"\[\[\s*EQUATION\s*\]\]",
        "[[EQUATION]]",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\[\[\s*CODE\s*\]\]",
        "[[CODE]]",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\[\[\s*CITATION\s*\]\]",
        "[[CITATION]]",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\[\[\s*COMPLEXITY\s*\]\]",
        "[[COMPLEXITY]]",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\[\[\s*URL\s*\]\]",
        "[[URL]]",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\[\[\s*FOREIGN\s*\]\]",
        "[[FOREIGN]]",
        text,
        flags=re.IGNORECASE
    )

    # Convert malformed equation brackets to one standard placeholder
    text = re.sub(
        r"\[{1,2}\s*EQUATION\s*\]{1,2}",
        "[[EQUATION]]",
        text,
        flags=re.IGNORECASE
    )

    return text


# Collapse repeated placeholders such as [[EQUATION]] [[EQUATION]]
def collapse_consecutive_placeholders(text):
    if not isinstance(text, str):
        return text

    pattern = rf"(?:{PLACEHOLDER_PATTERN}\s*){{2,}}"

    def replace_cluster(match):
        placeholders = re.findall(
            PLACEHOLDER_PATTERN,
            match.group(0)
        )

        if not placeholders:
            return ""

        return placeholders[0] + " "

    text = re.sub(
        pattern,
        replace_cluster,
        text
    )

    return text


# Collapse equation fragments that were split into multiple placeholders
def collapse_equation_fragments(text):
    if not isinstance(text, str):
        return text

    equation_fragment_pattern = re.compile(
        r"\[\[EQUATION\]\]"
        r"[^.!?\n]{0,2000}"
        r"\[\[EQUATION\]\]"
        r"[^.!?\n]{0,2000}",
        flags=re.IGNORECASE
    )

    def replace_equation_fragment(match):
        segment = match.group(0)

        math_indicators = 0
        math_indicators += segment.count("=")
        math_indicators += segment.count("+")
        math_indicators += segment.count("^")
        math_indicators += segment.count("_")
        math_indicators += segment.count("{")
        math_indicators += segment.count("}")

        if math_indicators >= 2:
            return " [[EQUATION]] "

        return segment

    text = equation_fragment_pattern.sub(
        replace_equation_fragment,
        text
    )

    # Catch cases such as [[EQUATION]] = [[EQUATION]] = [[EQUATION]]
    text = re.sub(
        r"\[\[EQUATION\]\]"
        r"(?:\s*[_^]\w+)?"
        r"(?:\s*[+=-]\s*"
        r"\[\[EQUATION\]\]"
        r"(?:\s*[_^]\w+)?)"
        r"+",
        " [[EQUATION]] ",
        text
    )

    return text


# Remove unwanted standalone symbols
def remove_symbol_noise(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"/\)",
        " ",
        text
    )

    text = re.sub(
        r"\(/",
        " ",
        text
    )

    text = re.sub(
        r"\(\)",
        " ",
        text
    )

    # Remove all asterisks
    text = text.replace(
        "*",
        " "
    )

    # Remove standalone slash
    text = re.sub(
        r"(?<!\w)/(?!\w)",
        " ",
        text
    )

    # Remove standalone parentheses
    text = text.replace(
        "(",
        " "
    )

    text = text.replace(
        ")",
        " "
    )

    return text


# Final pass to normalize the cleaned text
def final_text_cleanup(text):
    if not isinstance(text, str):
        return text

    text = normalize_placeholders(text)
    text = collapse_equation_fragments(text)
    text = clean_placeholder_parentheses(text)
    text = collapse_consecutive_placeholders(text)
    text = remove_symbol_noise(text)
    text = normalize_placeholders(text)
    text = collapse_equation_fragments(text)
    text = re.sub(
        r"\\+",
        " ",
        text
    )

    text = text.replace(
        "~",
        " "
    )

    text = re.sub(
        r"[{}]",
        " ",
        text
    )

    text = re.sub(
        r"(?<!\w)[_^]+(?!\w)",
        " ",
        text
    )

    # Convert [[EQUATION]] to the required [EQUATION] format
    text = re.sub(
        r"\[\[EQUATION\]\]",
        "[EQUATION]",
        text
    )

    # Make sure consecutive equation tags become one
    text = re.sub(
        r"(?:\s*\[EQUATION\]\s*){2,}",
        " [EQUATION] ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# Complete cleaning pipeline
def clean_mgtbench_pipeline(text):
    if not isinstance(text, str):
        return text

    text = text.replace(
        "\r",
        " "
    )

    text = text.replace(
        "\n",
        " "
    )

    text = text.replace(
        "\\n",
        " "
    )

    text = text.replace(
        "\\t",
        " "
    )

    text = clean_code_texts(text)
    text = clean_url(text)
    text = clean_foreign_script(text)
    text = clean_complexity_notation(text)
    text = strip_reference_list(text)
    text = clean_citations(text)
    text = clean_list_numbering(text)
    text = clean_math_texts(text)
    text = clean_residual_math_noise(text)
    text = final_text_cleanup(text)

    return text


# Clean the MGTBench AI dataset
def clean_mgtbench_ai_dataset(dataset):
    dataset = dataset.copy()
    original_rows = len(dataset)
    dataset["_before_being_cleaned"] = dataset["text"]
    dataset = dataset.dropna(
        subset=["text"]
    )
    dataset["text"] = (
        dataset["text"]
        .astype(str)
        .str.strip()
    )

    dataset = dataset[
        dataset["text"] != ""
    ]

    dataset["text"] = (
        dataset["text"]
        .apply(clean_mgtbench_pipeline)
    )

    dataset["text"] = (
        dataset["text"]
        .str.strip()
    )

    dataset = dataset[
        dataset["text"] != ""
    ]

    # Remove rows containing only placeholders
    only_placeholder = dataset["text"].str.fullmatch(
        rf"(?:{PLACEHOLDER_PATTERN}\s*)+"
    )

    only_equation = dataset["text"].str.fullmatch(
        r"\s*\[EQUATION\]\s*"
    )

    dataset = dataset[
        ~only_placeholder
    ]

    dataset = dataset[
        ~only_equation
    ]

    dataset = dataset.reset_index(
        drop=True
    )

    before_after = pd.DataFrame({
        "before_being_cleaned":
            dataset["_before_being_cleaned"],

        "after_text":
            dataset["text"]
    })

    before_after = (
        before_after
        .reset_index(drop=True)
    )

    dataset = dataset.drop(
        columns=["_before_being_cleaned"]
    )

    cleaning_stats = {
        "original_rows":
            original_rows,

        "cleaned_rows":
            len(dataset),

        "rows_removed":
            original_rows - len(dataset),

        "missing_values":
            dataset.isna()
            .sum()
            .to_dict(),

        "empty_texts":
            dataset["text"]
            .str.strip()
            .eq("")
            .sum(),

        "exact_duplicate_rows":
            dataset.duplicated()
            .sum(),

        "unique_files":
            dataset["file"].nunique()
            if "file" in dataset.columns
            else None
    }

    return (
        dataset,
        before_after,
        cleaning_stats
    )


# Run the cleaning using your existing mgtbench_ai dataset
mgtbench_ai_cleaned, mgtbench_before_after, cleaning_stats = (
    clean_mgtbench_ai_dataset(
        mgtbench_ai
    )
)
print("MGTBENCH AI CLEANING RESULTS")

print(
    f"\nOriginal rows: "
    f"{cleaning_stats['original_rows']:,}"
)

print(
    f"Cleaned rows:  "
    f"{cleaning_stats['cleaned_rows']:,}"
)

print(
    f"Rows removed:  "
    f"{cleaning_stats['rows_removed']:,}"
)

print(
    "\nEmpty texts after cleaning:"
)

print(
    cleaning_stats["empty_texts"]
)

print(
    "\nExact duplicate rows:"
)

print(
    cleaning_stats["exact_duplicate_rows"]
)

print(
    "\nUnique source files:"
)

print(
    cleaning_stats["unique_files"]
)


# Display before and after examples
print("\nBEFORE vs AFTER CLEANING")

ipd.display(
    mgtbench_before_after.head(20)
)


# Display cleaned dataset
print("\nCLEANED DATASET SAMPLE")

ipd.display(
    mgtbench_ai_cleaned.head(10)
)


# Check whether unwanted noise remains
print("\nFINAL NOISE CHECK")


remaining_backslashes = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        r"\\",
        regex=True,
        na=False
    )
    .sum()
)

remaining_tildes = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        "~",
        regex=False,
        na=False
    )
    .sum()
)

consecutive_equations = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        r"\[EQUATION\]\s+\[EQUATION\]",
        regex=True,
        na=False
    )
    .sum()
)

remaining_asterisks = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        r"\*",
        regex=True,
        na=False
    )
    .sum()
)

remaining_slash_parentheses = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        r"/\)",
        regex=True,
        na=False
    )
    .sum()
)

remaining_parentheses = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        r"[\(\)]",
        regex=True,
        na=False
    )
    .sum()
)


print(
    f"Texts containing '\\\\': "
    f"{remaining_backslashes}"
)

print(
    f"Texts containing '~': "
    f"{remaining_tildes}"
)

print(
    f"Texts with consecutive [EQUATION]: "
    f"{consecutive_equations}"
)

print(
    f"Texts containing '*': "
    f"{remaining_asterisks}"
)

print(
    f"Texts containing '/)': "
    f"{remaining_slash_parentheses}"
)

print(
    f"Texts containing parentheses: "
    f"{remaining_parentheses}"
)


# Show random before and after examples
print("\nRANDOM BEFORE vs AFTER EXAMPLES")


sample_size = min(
    10,
    len(mgtbench_ai_cleaned)
)


if sample_size > 0:

    sample_after = (
        mgtbench_ai_cleaned
        .sample(
            n=sample_size,
            random_state=42
        )
    )

    original_lookup = (
        mgtbench_ai
        .drop_duplicates("id")
        .set_index("id")["text"]
    )

    for i, (_, row) in enumerate(
        sample_after.iterrows(),
        start=1
    ):

        original_id = row["id"]

        if original_id in original_lookup.index:
            original_text = (
                original_lookup.loc[
                    original_id
                ]
            )
        else:
            original_text = (
                "[Original text not found]"
            )

        print(
            f"\nEXAMPLE {i}"
        )

        print(
            f"ID: {original_id}"
        )

        print(
            f"FILE: {row['file']}"
        )

        print(
            "\nBEFORE BEING CLEANED:"
        )

        print(
            original_text
        )

        print(
            "\nAFTER CLEANING:"
        )

        print(
            row["text"]
        )

        print(
            "\nCHANGED:"
        )

        print(
            "YES"
            if str(original_text)
            != str(row["text"])
            else "NO"
        )


# Find every row where the text changed
comparison = (
    mgtbench_ai_cleaned
    .copy()
)

comparison["before_being_cleaned"] = (
    comparison["id"]
    .map(
        mgtbench_ai
        .drop_duplicates("id")
        .set_index("id")["text"]
    )
)

comparison["after_text"] = (
    comparison["text"]
)

comparison["text_changed"] = (
    comparison["before_being_cleaned"]
    != comparison["after_text"]
)

changed_rows = comparison[
    comparison["text_changed"]
].copy()


print("\nCHANGED TEXTS")

print(
    f"\nRows with text changes: "
    f"{len(changed_rows):,}"
)

ipd.display(
    changed_rows[
        [
            "id",
            "file",
            "before_being_cleaned",
            "after_text"
        ]
    ].head(20)
)


# Prepare the before/after output
before_after_output = changed_rows[
    [
        "id",
        "file",
        "before_being_cleaned",
        "after_text"
    ]
].copy()

print(
    "\nBefore/after comparison is ready."
)

ipd.display(
    before_after_output.head(10)
)

MGTBENCH AI CLEANING RESULTS

Original rows: 313,775
Cleaned rows:  313,619
Rows removed:  156

Empty texts after cleaning:
0

Exact duplicate rows:
0

Unique source files:
46

BEFORE vs AFTER CLEANING


,before_being_cleaned,after_text
0,"In this report, we present and compare two met...","In this report, we present and compare two met..."
1,The results presented in this study shed light...,The results presented in this study shed light...
2,Kr sensitivities and uncertainties\n\nIn this ...,Kr sensitivities and uncertainties In this sec...
3,"In this section, we aim to quantify the non-Ga...","In this section, we aim to quantify the non-Ga..."
4,The key innovation of this study is the replac...,The key innovation of this study is the replac...
5,\\section{Results and Discussion}\n\nOur resea...,[EQUATION] Our research emphasizes the importa...
6,Motion of stars relative to their local inters...,Motion of stars relative to their local inters...
7,IceCube studies a diverse range of physics top...,IceCube studies a diverse range of physics top...
8,The observed spectral energy distribution (SED...,The observed spectral energy distribution SED ...
9,We established the photometric catalogs in eac...,We established the photometric catalogs in eac...



CLEANED DATASET SAMPLE


,id,text,file
0,0,"In this report, we present and compare two met...",../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
1,2,The results presented in this study shed light...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
2,4,Kr sensitivities and uncertainties In this sec...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
3,5,"In this section, we aim to quantify the non-Ga...",../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
4,6,The key innovation of this study is the replac...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
5,7,[EQUATION] Our research emphasizes the importa...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
6,8,Motion of stars relative to their local inters...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
7,9,IceCube studies a diverse range of physics top...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
8,10,The observed spectral energy distribution SED ...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
9,11,We established the photometric catalogs in eac...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...



FINAL NOISE CHECK
Texts containing '\\': 0
Texts containing '~': 0
Texts with consecutive [EQUATION]: 0
Texts containing '*': 0
Texts containing '/)': 0
Texts containing parentheses: 0

RANDOM BEFORE vs AFTER EXAMPLES

EXAMPLE 1
ID: 1352
FILE: Electrical_engineering_wiki_new.json

BEFORE BEING CLEANED:
In the early universe, Eqs.~\\eqref{dq} are slightly modified. While the equation of motion for $\\mathbf{D}$ remains unchanged, the second line becomes $\\dot{\\mathbf{Q}} = \\mu \\mathbf{D} \\times \\mathbf{Q} - 4 H \\left( \\omega / \\mu \\right) \\mathbf{B}$, where $H$ represents the Hubble constant. Despite the time-dependence induced by the universe's expansion, certain quantities remain strictly conserved. Specifically, $\\mathbf{B} \\cdot \\mathbf{D}$ maintains its value, which can be interpreted as the angular momentum along the gravitational field. Additionally, $\\mathbf{D} \\cdot \\mathbf{Q} + \\frac{\\omega}{\\mu} \\mathbf{B} \\cdot \\mathbf{D}$ remains constant and upholds

,id,file,before_being_cleaned,after_text
0,0,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,"In this report, we present and compare two met...","In this report, we present and compare two met..."
1,2,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The results presented in this study shed light...,The results presented in this study shed light...
2,4,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,Kr sensitivities and uncertainties\n\nIn this ...,Kr sensitivities and uncertainties In this sec...
3,5,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,"In this section, we aim to quantify the non-Ga...","In this section, we aim to quantify the non-Ga..."
4,6,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The key innovation of this study is the replac...,The key innovation of this study is the replac...
5,7,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,\\section{Results and Discussion}\n\nOur resea...,[EQUATION] Our research emphasizes the importa...
6,8,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,Motion of stars relative to their local inters...,Motion of stars relative to their local inters...
7,9,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,IceCube studies a diverse range of physics top...,IceCube studies a diverse range of physics top...
8,10,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The observed spectral energy distribution (SED...,The observed spectral energy distribution SED ...
9,11,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,We established the photometric catalogs in eac...,We established the photometric catalogs in eac...



Before/after comparison is ready.


,id,file,before_being_cleaned,after_text
0,0,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,"In this report, we present and compare two met...","In this report, we present and compare two met..."
1,2,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The results presented in this study shed light...,The results presented in this study shed light...
2,4,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,Kr sensitivities and uncertainties\n\nIn this ...,Kr sensitivities and uncertainties In this sec...
3,5,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,"In this section, we aim to quantify the non-Ga...","In this section, we aim to quantify the non-Ga..."
4,6,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The key innovation of this study is the replac...,The key innovation of this study is the replac...
5,7,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,\\section{Results and Discussion}\n\nOur resea...,[EQUATION] Our research emphasizes the importa...
6,8,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,Motion of stars relative to their local inters...,Motion of stars relative to their local inters...
7,9,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,IceCube studies a diverse range of physics top...,IceCube studies a diverse range of physics top...
8,10,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The observed spectral energy distribution (SED...,The observed spectral energy distribution SED ...
9,11,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,We established the photometric catalogs in eac...,We established the photometric catalogs in eac...


In [9]:
def clean_bawe_corpus_dataset(dataset):

    pass